# Modelling with Pipelines

![Status](https://img.shields.io/static/v1.svg?label=Status&message=Finished&color=brightgreen)
[![Source](https://img.shields.io/static/v1.svg?label=GitHub&message=Source&color=181717&logo=GitHub)](https://github.com/particle1331/ok-transformer/blob/master/docs/nb/fundamentals/pipelines.ipynb)
[![Stars](https://img.shields.io/github/stars/particle1331/ok-transformer?style=social)](https://github.com/particle1331/ok-transformer)


---

## Introduction

Pipelines in the [`scikit-learn` API](https://scikit-learn.org/stable/modules/classes.html#) allow us to apply a sequence of transformers and a final estimator on our dataset. Since each intermediate step implements a `fit` and `transform` method, while the final estimator implements a `fit` method, this setup allows us to recursively fit of all transformations on the dataset in a single step. Another advantage is that the pipeline can be cross-validated as a whole while setting different parameters. 

From the perspective of code maintainability and testability, pipelines are easier to work with because they allow us to separate declarative from imperative code as shown in {numref}`pipelines`. In the last section of this notebook, we will show that we can easily load pipelines to make inference on test data with little to no preprocessing.

```{figure} ../../img/pipelines.png
---
width: 40em
name: pipelines
---
Pipelines allow us to cleanly separate declarative from imperative code. [[source]](https://gh.mltrainings.ru/presentations/LopuhinJankiewicz_KaggleMercari.pdf)
```

In this notebook, we make heavy use of the [`feature-engine`](https://feature-engine.readthedocs.io/en/latest/index.html) library for our feature transformations. Feature-engine allows you to select the variables you want to transform within each transformer. This way different feature engineering procedures can be easily applied to different feature subsets resulting in clean pipeline code.

## House prices dataset

In this notebook, we will use the dataset from the [House Prices - Advanced](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) competition in Kaggle. This is a dataset with 79 features describing almost every aspect of residential homes in Ames, Iowa. We will use these features to predict the final price of each home. 

Downloading the dataset:

```{margin}
`kaggle/v1.5.12`
```

```bash
COMPETITION=house-prices-advanced-regression-techniques
DATA_DIR=./data
mkdir ${DATA_DIR}

kaggle competitions download -c ${COMPETITION} -p ${DATA_DIR}
unzip ${DATA_DIR}/${COMPETITION}.zip -d ${DATA_DIR}/${COMPETITION} > /dev/null
rm ${DATA_DIR}/${COMPETITION}.zip
```

```tex
Downloading house-prices-advanced-regression-techniques.zip to ./data
100%|███████| 199k/199k [00:00<00:00, 1.67MB/s]
100%|███████| 199k/199k [00:00<00:00, 1.66MB/s]
```

Loading the data:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats('svg')


data = pd.read_csv('data/house-prices-advanced-regression-techniques/train.csv')
data.drop(['Id'], axis=1, inplace=True)

print(data.shape)
data.head()

In [ ]:
data.columns

Identifying categorical and numerical features:

In [ ]:
categorical = [f for f in data.columns if data[f].dtype == 'O'] + ['MSSubClass']
data[categorical] = data[categorical].astype('O')

numerical = [f for f in data.columns if f not in categorical and f != 'SalePrice']

### Target distribution

Normalizing the target distribution:

In [ ]:
import warnings
warnings.simplefilter(action='ignore')

fig, ax = plt.subplots(1, 2, figsize=(8, 3))

sns.distplot(data.SalePrice.values, bins=50, ax=ax[0])
sns.distplot(np.log(data.SalePrice.values), bins=50, ax=ax[1]);

ax[0].set_xlabel('Sale Price')
ax[1].set_xlabel('Log Sale Price');

Log-transforming the target:

In [ ]:
data['LogSalePrice'] = np.log(data['SalePrice'])

### Missing values

Percentage of examples with missing feature:

In [ ]:
var_with_nan = [v for v in data.columns if data[v].isnull().sum() > 0]
percentage_nan = data[var_with_nan].isnull().mean().sort_values(ascending=True)

# Plot
percentage_nan.plot.bar(figsize=(8, 4), color='#404040')
plt.ylabel('% missing')
plt.axhline(y=0.80, color='#FF0000', linestyle='--', linewidth=0.7)
plt.show()

In [ ]:
categorical_na = [v for v in categorical if v in var_with_nan]
numerical_na = [v for v in numerical if v in var_with_nan]

print(f'No. categorical w/ nans: {len(categorical_na)}/{len(categorical)}')
print(f'No. numerical w/ nans:    {len(numerical_na)}/{len(numerical)}')

Effect of missingness of categorical features on sale price:

In [ ]:
data_ = data.copy()
for v in data_.columns:
    data_[f"{v}_isna"] = data_[v].isna().astype(int)

fig, ax = plt.subplots(4, 4, figsize=(10, 8))
k = 0
for i in range(4):
    for j in range(4):
        sns.histplot(data_.query(f"{categorical_na[k]}_isna == 0").LogSalePrice.values, bins=20, ax=ax[i, j], color="C0", stat="density", kde=True)
        sns.histplot(data_.query(f"{categorical_na[k]}_isna == 1").LogSalePrice.values, bins=20, ax=ax[i, j], color="C1", stat="density", kde=True)
        ax[i, j].set_title(f"{categorical_na[k]}", size=10)
        if j > 0:
            ax[i, j].set_ylabel("")
        k += 1

fig.suptitle("Density of log sale price whenever a feature is missing (orange) or present (blue)")
fig.tight_layout();

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(6, 2))
k = 0
for i in range(3):
    sns.histplot(data_.query(f"{numerical_na[k]}_isna == 0").LogSalePrice.values, bins=20, ax=ax[i], color="C0", stat="density", kde=True)
    sns.histplot(data_.query(f"{numerical_na[k]}_isna == 1").LogSalePrice.values, bins=20, ax=ax[i], color="C1", stat="density", kde=True)
    ax[i].set_title(f"{numerical_na[k]}", size=10)
    if i > 0:
        ax[i].set_ylabel("")
    k += 1
    
fig.suptitle("Distribution of log sale price whenever a feature is missing (orange) or present (blue)")
fig.tight_layout()

### Temporal features

Getting year features:

In [ ]:
year_features = [f for f in numerical if 'Yr' in f or 'Year' in f]
year_features

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3))

price_yrsold = data.groupby('YrSold')['LogSalePrice'].median()
ax[0].plot(price_yrsold)
ax[0].set_xticks(price_yrsold.index.astype(int))
ax[0].set_ylabel('Median House Price')
ax[0].set_xlabel("Year Sold")

price_yrbuilt = data.groupby('YearBuilt')['LogSalePrice'].median()
ax[1].plot(price_yrbuilt)
ax[1].set_ylabel("")
ax[1].set_xlabel("Year Built")
ax[1].axhline(price_yrsold.max(), color='gray', linestyle='dashed', linewidth=0.8)
ax[1].axhline(price_yrsold.min(), color='gray', linestyle='dashed', linewidth=0.8)
ax[1].set_xlim(price_yrbuilt.index.min() - 10, price_yrbuilt.index.max() + 10)
ax[1].fill_between(x=[1800]+list(price_yrbuilt.index)+[2050], y1=price_yrsold.min(), y2=price_yrsold.max(), color='gray', alpha=0.5)

fig.tight_layout()

The range of values of sold houses are plotted in the right plot. This shows that only houses pre-1990s are being sold between 2006 to 2010. Next we plot the average years from making builds on the house (including the initial build) to it being sold. Here we see mostly that houses are being bought within 15 years of these constructions.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(6, 8))

sns.distplot(data.YrSold - data.YearBuilt,    ax=ax[0], bins=30, label="Built")
sns.distplot(data.YrSold - data.YearRemodAdd, ax=ax[0], bins=30, label="RemodAdd")
sns.distplot(data.YrSold - data.GarageYrBlt,  ax=ax[0], bins=30, label="Garage")
ax[0].set_xlabel("Years sold after")
ax[0].set_xlim(-25, 160)
ax[0].legend()

median_saleprice = lambda year_var: data_.groupby(f"{year_var}_gap").LogSalePrice.median()

for year_var in ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt']:
    data_[f"{year_var}_gap"] = data.YrSold - data[year_var].values
    ax[1].plot(median_saleprice(year_var), label=year_var)

ax[1].set_xlabel("Years sold after")
ax[1].set_ylabel("Median log sale price")
ax[1].set_xlim(-25, 160)
ax[1].legend();

### Discrete features

For our purposes, we will define discrete variables as numeric variables with less than 20 unique values that are not year variables.

In [ ]:
discrete = [v for v in numerical if (data[v].nunique() < 20) and (v not in year_features)]

print('No. discrete features:', len(discrete))
data[discrete].head()

These tend to be grading scales, so we expect increasing trend with `SalePrice`. Note that the [violin plots](https://images.ctfassets.net/fi0zmnwlsnja/sdfgtcRp16wTNOcRceGQm/5bfcb73d2261d49ff20dd7857e0152b1/Screen_Shot_2019-03-01_at_11.36.10_AM.png) are nicely scaled based on count.

In [ ]:
def plot_violin_vs_logsaleprice(data, columns, rows, cols, row_size, col_size, rotation=60):
    fig, ax = plt.subplots(rows, cols, figsize=(cols * col_size, rows * row_size))
    k = 0
    for i in range(rows):
        for j in range(cols):
            if k < len(columns):
                v = columns[k]
                sns.violinplot(
                    ax=ax[i, j], 
                    x=v, 
                    y='LogSalePrice', 
                    data=data, 
                    height=4, 
                    aspect=1.5, 
                    scale='count'
                )
                if j == 0:
                    ax[i, j].set_ylabel("LogSalePrice")
                else:
                    ax[i, j].set_ylabel("")
                ax[i, j].set_xlabel("")
                ax[i, j].set_title(v)
                ax[i, j].tick_params(axis='x', labelrotation=rotation)
            else:
                ax[i, j].set_visible(False)
            k += 1

    fig.tight_layout()


plot_violin_vs_logsaleprice(data=data, columns=discrete, rows=5, cols=3, row_size=3, col_size=3.5, rotation=0)

### Continuous features

For our purposes, continuous features are numerical features that are neither years nor discrete.

In [ ]:
continuous = [v for v in numerical if v not in discrete + year_features]
print('Number of continuous variables: ', len(continuous))

In [ ]:
data[continuous].head()

In [ ]:
fig, ax = plt.subplots(5, 4, figsize=(10, 8))
k = 0
for i in range(5):
    for j in range(4):
        if k < 18:
            v = continuous[k]
            sns.histplot(ax=ax[i, j], data=data[v], bins=30, stat='density', kde=True)
            ax[i, j].set_ylabel("")
        else:
            ax[i, j].set_visible(False)
        k += 1

fig.suptitle("Distributions of continuous feature values.")
fig.tight_layout()

#### Transforms

Note that the data looks mostly left-skewed. To remedy this, we can transform the data using monotonic transformations. For left-skewed positive variables, we can use the log-transform. Another transformation that works with nonnegative variables is to square the variable. Both these transformations work by stretching out large values of the variable. 

An automatic [power transform](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PowerTransformer.html) that supports both negative and positive values is called **Yeo-Johnson** which we will use below. This is unlikely to work well with extremely skewed variables. So we  further restrict continuous variables to not include these variables.

In [ ]:
extreme_skewed = [
    'BsmtFinSF2', 
    'LowQualFinSF', 
    'EnclosedPorch',
    '3SsnPorch', 
    'ScreenPorch', 
    'MiscVal'
]

# Update continuous features
continuous = [c for c in continuous if c not in extreme_skewed]

Checking if Yeo-Johnson works well:

In [ ]:
import scipy

def compare_transformed_dist(f, data, transform_name, columns, rows, cols, row_size, col_size):
    data_transformed = data.copy()
    data_orig_scaled = data.copy() # Original data scaled

    for v in columns:
        # Transform the variable using Yeo-Johnson
        data_transformed[v], _ = f(data[v])
        data_transformed[v] = (data_transformed[v] - data_transformed[v].min()) / (data_transformed[v].max() - data_transformed[v].min())  # scale
        data_orig_scaled[v] = (data[v] - data[v].min()) / (data[v].max() - data[v].min()) 

    # Plot the histograms of the transformed variables
    fig, ax = plt.subplots(rows, cols, figsize=(cols * col_size, rows * row_size))
    k = 0
    for i in range(rows):
        for j in range(cols):
            if k < rows * cols:
                v = columns[k]
                sns.histplot(ax=ax[i, j], data=data_orig_scaled[v], bins=30, kde=True, stat='density', color="C0");
                sns.histplot(ax=ax[i, j], data=data_transformed[v], bins=30, kde=True, stat='density', color="C1")
                ax[i, j].set_ylabel("")
            else:
                ax[i, j].set_visible(False)
            k += 1

    fig.suptitle(f"Distribution of {transform_name} transformed (orange) and original feature (blue).")
    fig.tight_layout()

    return data_transformed, data_orig_scaled


data_transformed, data_orig_scaled = compare_transformed_dist(
    f=scipy.stats.yeojohnson, 
    data=data,
    transform_name='Yeo-Johnson',
    columns=continuous, rows=4, cols=3,
    row_size=2, col_size=3
)

For `LotFrontage` and `MasVnrArea` the transformation did not do an amazing job. Otherwise, the distribution looks better. Whether this helps improve the predictive power, remains to be seen. To determine if this is the case, we should train a model with the original values and one with the transformed values, and determine model performance, and feature importance. Here we do a quick visualization.

In [ ]:
def compare_transformed_price(data_transformed, data_orig_scaled, transform_name, columns, rows, cols, row_size=2.5, col_size=3.3):
    # Plot the histograms of the transformed variables
    fig, ax = plt.subplots(rows, cols, figsize=(rows * row_size, cols * col_size))
    k = 0
    for i in range(rows):
        for j in range(cols):
            if k < rows * cols:
                v = columns[k]
                sns.regplot(x=v, y="LogSalePrice", data=data_orig_scaled, ax=ax[i, j], color="C0", label="Original")
                sns.regplot(x=v, y="LogSalePrice", data=data_transformed, ax=ax[i, j], color="C1", label="Transformed", marker='x')
            else:
                ax[i, j].set_visible(False)
            if j > 0:
                ax[i, j].set_ylabel("")
            k += 1

    fig.suptitle(f"LogSalePrice vs {transform_name} transformed (orange) and original (blue) feature values.")
    fig.tight_layout()

compare_transformed_price(data_transformed, data_orig_scaled, 'Yeo-Johnson', columns=continuous, rows=4, cols=3)

By eye, the transformations seems to improve the relationship only for `LotArea` and `GrLivArea`. For features with positive values, let us check if log-transform would work better.

In [ ]:
# Need to make log-transform to 2-valued (like Yeo-Johnson)
@np.vectorize
def log(x):
    return np.log(x), 0

positive = [c for c in continuous if data[c].min() > 0]
data_transformed, data_orig_scaled = compare_transformed_dist(
    f=log, 
    data=data,
    transform_name='log',
    columns=positive, 
    rows=2, cols=2,
    row_size=2, col_size=3
)

The transformed features appear to have nicer distributions. Let us check its spread of values against with the target value.

In [ ]:
compare_transformed_price(
    data_transformed, data_orig_scaled, 
    transform_name='log',
    columns=positive, rows=2, cols=2,
    row_size=3, col_size=3
)

It turns out that the transformed variables have a better spread of values which may help models make better predictions.

#### Extremely skewed features

These variables are mostly zero:

In [ ]:
(data[extreme_skewed] == 0).mean()

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(10, 6))
k = 0
for i in range(2):
    for j in range(3):
        v = extreme_skewed[k]
        sns.histplot(data[data[v] == 0].LogSalePrice.values, bins=20, ax=ax[i, j], color="C0", stat="density", kde=True)
        sns.histplot(data[data[v] >  0].LogSalePrice.values, bins=20, ax=ax[i, j], color="C1", stat="density", kde=True)
        ax[i, j].set_title(v, size=10)
        if j > 0:
            ax[i, j].set_ylabel("")
        k += 1

fig.suptitle("Distribution of skewed feature if nonzero (orange) or zero (blue)")
fig.tight_layout();

### Categorical features

In [ ]:
print('No. categorical variables: ', len(categorical))

In [ ]:
data[categorical].head()

In [ ]:
data[categorical].nunique().sort_values(ascending=False).plot.bar(figsize=(12,5));

#### Quality features

All the categorical variables show low cardinality, this means that they have only few different labels which is good. The following encodings are hand-crafted based on the metadata. This can be captured automatically using ordered label encoding for example. For example:

```text
FireplaceQu: Fireplace quality

       Ex	Excellent - Exceptional Masonry Fireplace
       Gd	Good - Masonry Fireplace in main level
       TA	Average - Prefabricated Fireplace in main living area or Masonry Fireplace in basement
       Fa	Fair - Prefabricated Fireplace in basement
       Po	Poor - Ben Franklin Stove
       NA	No Fireplace
```

Applying the mappings:

In [ ]:
data_transformed = data.copy()

# ==============================================================================

qual_mappings = {
    'Po': 1, 
    'Fa': 2, 
    'TA': 3, 
    'Gd': 4, 
    'Ex': 5, 
    'Missing': 0, 
    'NA': 0
}

qual_vars = [
    'ExterQual', 
    'ExterCond', 
    'BsmtQual', 
    'BsmtCond',
    'HeatingQC', 
    'KitchenQual', 
    'FireplaceQu',
    'GarageQual', 
    'GarageCond',
]

for v in qual_vars:
    data_transformed[v] = data[v].fillna('NA').map(qual_mappings)

# ==============================================================================

exposure_mappings = {
    'No': 1, 
    'Mn': 2, 
    'Av': 3, 
    'Gd': 4, 
    'Missing': 0, 
    'NA': 0
}

exposure_vars = ['BsmtExposure']

for v in exposure_vars:
    data_transformed[v] = data[v].fillna('NA').map(exposure_mappings)

# ==============================================================================

finish_mappings = {
    'Missing': 0, 
    'NA': 0, 
    'Unf': 1, 
    'LwQ': 2, 
    'Rec': 3, 
    'BLQ': 4, 
    'ALQ': 5, 
    'GLQ': 6
}

finish_vars = ['BsmtFinType1', 'BsmtFinType2']
for v in finish_vars:
    data_transformed[v] = data[v].fillna('NA').map(finish_mappings)

# ==============================================================================

garage_mappings = {
    'Missing': 0, 
    'NA': 0, 
    'Unf': 1, 
    'RFn': 2, 
    'Fin': 3
}

garage_vars = ['GarageFinish']
for v in garage_vars:
    data_transformed[v] = data[v].fillna('NA').map(garage_mappings)

# ==============================================================================

fence_mappings = {
    'Missing': 0, 
    'NA': 0, 
    'MnWw': 1, 
    'GdWo': 2, 
    'MnPrv': 3, 
    'GdPrv': 4
}

fence_vars = ['Fence']
for v in fence_vars:
    data_transformed[v] = data[v].fillna('NA').map(fence_mappings)

# ==============================================================================
# All quality variables
quality  = qual_vars + finish_vars + exposure_vars + garage_vars + fence_vars
print(len(quality))

In [ ]:
plot_violin_vs_logsaleprice(data=data_transformed, columns=quality, rows=5, cols=3, row_size=2, col_size=3.5, rotation=0)

For most attributes, the increase in the house price with the value of the variable is quite clear.

In [ ]:
categorical_others = [v for v in categorical if v not in quality]
len(categorical_others)

#### Rare labels

For our purposes, we define rare as belonging to only 1% of observations.

In [ ]:
data_copy = data.copy()
rare_transforms = {}

for v in categorical_others:
    rare_threshold = 0.01
    rare_percentage = data_copy[v].value_counts(normalize=True)
    rare_values = rare_percentage[rare_percentage < rare_threshold]
    
    print(f"Rare values: {len(rare_values)}/{data_copy[v].nunique()}")
    print(rare_values)
    print()

    rare_values = list(rare_values.index)
    rare_transforms[v] = dict(zip(rare_values, ['Rare'] * len(rare_values)))
    for u in [_ for _ in data[v].unique() if _ not in rare_values]:
        rare_transforms[v][u] = u

    data_copy[v] = data_copy[v].map(rare_transforms[v])

Some of the categorical variables show multiple **rare labels** that are present in less than 1% of the houses. Labels that are under-represented in the dataset tend to cause over-fitting of machine learning models. That is why we want to remove them or collect them in one label `rare`.

#### Other categorical features

Finally, we want to explore the relationship between the categories of the different variables and the house sale price:

In [ ]:
plot_violin_vs_logsaleprice(data=data_copy, columns=categorical_others, rows=10, cols=3, row_size=3, col_size=5)

## Feature engineering

In this section, we create a pipeline for feature engineering based on the EDA above.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, Binarizer
from sklearn.base import BaseEstimator, TransformerMixin

from feature_engine.selection import DropFeatures
from feature_engine.wrappers import SklearnTransformerWrapper
from feature_engine.imputation import (
    AddMissingIndicator,
    MeanMedianImputer,
    CategoricalImputer
)
from feature_engine.encoding import (
    RareLabelEncoder,
    OrdinalEncoder
)
from feature_engine.transformation import (
    LogTransformer,
    YeoJohnsonTransformer
)

RANDOM_STATE = 0

Our feature engineering techniques will learn things like mean, mode, exponents for the Yeo-Johnson transforms, category frequency, and category to integer mappings. So first we need to have a validation set. Note that our target will be the logarithm of the sale price.


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    data.drop(['LogSalePrice', 'SalePrice'], axis=1), 
    data['LogSalePrice'], 
    test_size=0.1, 
    random_state=RANDOM_STATE,
)

X_train.shape, X_valid.shape

### Temporal features

For transforming temporal variables, we define custom transformers. Note that all rows have `YrSold` feature. Otherwise, we have to drop it since this feature is required along with `SalePrice`. Luckily, looks like the data is clean in this regard.

In [ ]:
X_train.YrSold.isna().sum()

In [ ]:
class TemporalVariableTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, variables, reference_variable):
        """Transform time to elapsed time from reference variable."""

        if not isinstance(variables, list):
            raise ValueError('variables should be a list')
        
        self.variables = variables
        self.reference_variable = reference_variable

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_ = X.copy()        
        for feature in self.variables:
            X_[feature] = X_[self.reference_variable] -  X_[feature]
        return X_


yr_features = ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt']
yr_transformer = TemporalVariableTransformer(variables=yr_features, reference_variable='YrSold')
yr_transformer.fit(X_train)
X_train = yr_transformer.transform(X_train)
X_valid = yr_transformer.transform(X_valid)

yr_dropper = DropFeatures(features_to_drop=['YrSold'])
yr_dropper.fit(X_train)
X_train = yr_dropper.transform(X_train)
X_valid = yr_dropper.transform(X_valid)

Looks good:

In [ ]:
X_train[yr_features].head(3)

In [ ]:
X_valid[yr_features].head(3)

### Discrete features

The discrete variables already look well behaved. Also no missing values:

In [ ]:
X_train[discrete].isna().sum()

### Continuous features

#### Missing continuous features

Recall that we have three numerical features with missing values. For these we will add indicator variables and fill the missing values with the mean.

In [ ]:
missing_indicator = AddMissingIndicator(variables=numerical_na)
missing_indicator.fit(X_train)
X_train = missing_indicator.transform(X_train)
X_valid = missing_indicator.transform(X_valid)

impute_mean = MeanMedianImputer(imputation_method='mean', variables=numerical_na)
impute_mean.fit(X_train)
X_train = impute_mean.transform(X_train)
X_valid = impute_mean.transform(X_valid)

In [ ]:
X_train[numerical_na + [f"{v}_na" for v in numerical_na]].query("LotFrontage_na==1 or MasVnrArea_na==1 or GarageYrBlt_na==1").head(10)

In [ ]:
X_valid[numerical_na + [f"{v}_na" for v in numerical_na]].query("LotFrontage_na==1 or MasVnrArea_na==1 or GarageYrBlt_na==1").head(10)

#### Transforms

Next we look at skewness. Recall that log-transform worked quite well for all positive numerical features and Yeo-Johnson only worked for `LotArea` and `GLivArea`.

In [ ]:
log_transformer = LogTransformer(variables=positive)
log_transformer.fit(X_train)
X_train = log_transformer.transform(X_train)
X_valid = log_transformer.transform(X_valid)

yeo_transformer = YeoJohnsonTransformer(variables=['LotArea', 'GrLivArea'])
yeo_transformer.fit(X_train)
X_train = yeo_transformer.transform(X_train)
X_valid = yeo_transformer.transform(X_valid)

#### Extremely skewed features

For extremely skewed feature, we binarize it. This is because, the model will have difficulty learning the sparse nonzero values of the features.

In [ ]:
# Binarizer(threshold=0): x = 0 if x <= 0, else x = 1.
binarizer = SklearnTransformerWrapper(transformer=Binarizer(threshold=0), variables=extreme_skewed)
binarizer.fit(X_train)
X_train = binarizer.transform(X_train)
X_valid = binarizer.transform(X_valid)

Plotting:

In [ ]:
X_valid[extreme_skewed].hist()
plt.tight_layout();

### Categorical features

#### Missing categorical features

Recall that missing values for categorical features and predictive of the sale price. Hence, we can introduce a new category for it. For features with lots of missing data, we replace the missing values with a new category `"Missing"`. For features with little missing data, we don't want to add unnecessary noise, so we instead replace nans with most frequent category in features that contain fewer observations with missing values.

In [ ]:
X_train[categorical_na].isna().mean().sort_values(ascending=False)

In [ ]:
X_train['MasVnrType'].value_counts(dropna=False)

In [ ]:
X_train['PoolQC'].value_counts(dropna=False)

Training the categorical imputer:

In [ ]:
replace_missing = [v for v in categorical_na if X_train[v].isnull().mean() > 0.1]
replace_frequent = [v for v in categorical_na if v not in replace_missing]

fill_missing = CategoricalImputer(imputation_method="missing", variables=replace_missing)
fill_missing.fit(X_train)
X_train = fill_missing.transform(X_train)
X_valid = fill_missing.transform(X_valid)

fill_frequent = CategoricalImputer(imputation_method="frequent", variables=replace_frequent)
fill_frequent.fit(X_train)
X_train = fill_frequent.transform(X_train)
X_valid = fill_frequent.transform(X_valid)

Checking results:

In [ ]:
# Nans are replaced by frequent category 'None'
X_train['MasVnrType'].value_counts(dropna=False)

In [ ]:
# Nans replaced by new 'Missing' category
X_train['PoolQC'].value_counts(dropna=False)

#### Quality mappings

Quality categorical features behave the same way as discrete numeric features. Here we just do the same as in the EDA section above and encode categories based using dictionaries from the metadata. The only difference is that we use transformers from `feature-engine`. 

In [ ]:
class Mapper(BaseEstimator, TransformerMixin):
    def __init__(self, variables, mappings):
        """Categorical missing value imputer"""

        if not isinstance(variables, list):
            raise ValueError('variables should be a list')

        self.variables = variables
        self.mappings = mappings

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_ = X.copy()
        for feature in self.variables:
            X_[feature] = X_[feature].map(self.mappings)
        return X_


mapper_qual = Mapper(variables=qual_vars, mappings=qual_mappings)
mapper_qual.fit(X_train)
X_train = mapper_qual.transform(X_train)
X_valid = mapper_qual.transform(X_valid)

mapper_exposure = Mapper(variables=exposure_vars, mappings=exposure_mappings)
mapper_exposure.fit(X_train)
X_train = mapper_exposure.transform(X_train)
X_valid = mapper_exposure.transform(X_valid)

mapper_finish = Mapper(variables=finish_vars, mappings=finish_mappings)
mapper_finish.fit(X_train)
X_train = mapper_finish.transform(X_train)
X_valid = mapper_finish.transform(X_valid)

mapper_garage = Mapper(variables=garage_vars, mappings=garage_mappings)
mapper_garage.fit(X_train)
X_train = mapper_garage.transform(X_train)
X_valid = mapper_garage.transform(X_valid)

mapper_fence = Mapper(variables=fence_vars, mappings=fence_mappings)
mapper_fence.fit(X_train)
X_train = mapper_fence.transform(X_train)
X_valid = mapper_fence.transform(X_valid)

Checking results:

In [ ]:
X_train[quality].head(3)

In [ ]:
X_valid[quality].head(3)

#### Rare labels

Next we map rare labels to a new category. Recall that we mapped categories which occur only on 1% of the data using a new category `"Rare"`. We applied this to `categorical_others` which are categorical variables that are not quality variables (i.e. those that already has some relationship with `SalePrice` based on domain knowledge).

In [ ]:
X_train[categorical_others] = X_train[categorical_others].astype('O')
X_valid[categorical_others] = X_valid[categorical_others].astype('O')

rare_encoder = RareLabelEncoder(tol=0.01, n_categories=1, variables=categorical_others)
rare_encoder.fit(X_train)
X_train = rare_encoder.transform(X_train)
X_valid = rare_encoder.transform(X_valid)

Here `n_categories=1` means that a feature must have 1 categories for the encoder to find frequent labels. Of course, all features have this trivial property.

#### Target encoding

Finally, we target encode the other categorical features which includes the added rare category for each feature that we added above. Note that having `encoding_method='ordered'` means that the categories are numbered in ascending order according to the target mean value per category.

In [ ]:
target_encoder = OrdinalEncoder(encoding_method='ordered', variables=categorical_others)
target_encoder.fit(X_train, y_train) # Provide features and targets
X_train = target_encoder.transform(X_train)
X_valid = target_encoder.transform(X_valid)

The resulting train and validation sets can then be used to train and validate models. In the next section, we look at a cleaner and more maintainable way of coding this. Note that we have to have features on the same scale if we are to use a linear regression model.

## Model pipeline

Note the above process of fitting each transformer and incrementally transforming the dataset is repetitive and error prone. Instead, we can arrange the whole feature engineering process into a single pipeline that has a `fit` and `transform` method. Moreover, we can add a model at the end for making predictions. Note that the final pipeline exposes the methods (e.g. `predict`) and attributes of the final model.

In [ ]:
# Temporal transforms
yr_transformer = TemporalVariableTransformer(variables=yr_features, reference_variable='YrSold')
yr_dropper = DropFeatures(features_to_drop=['YrSold'])

# Continuous transforms
missing_indicator = AddMissingIndicator(variables=numerical_na)
impute_mean = MeanMedianImputer(imputation_method='mean', variables=numerical_na)
log_transformer = LogTransformer(variables=positive)
yeo_transformer = YeoJohnsonTransformer(variables=['LotArea', 'GrLivArea'])
binarizer = SklearnTransformerWrapper(transformer=Binarizer(threshold=0), variables=extreme_skewed)

# Missing categorical features
fill_missing = CategoricalImputer(imputation_method="missing", variables=replace_missing)
fill_frequent = CategoricalImputer(imputation_method="frequent", variables=replace_frequent)

# Categorical quality features
mapper_qual = Mapper(variables=qual_vars, mappings=qual_mappings)
mapper_exposure = Mapper(variables=exposure_vars, mappings=exposure_mappings)
mapper_finish = Mapper(variables=finish_vars, mappings=finish_mappings)
mapper_garage = Mapper(variables=garage_vars, mappings=garage_mappings)
mapper_fence = Mapper(variables=fence_vars, mappings=fence_mappings)

# Categorical rare and target encoding
rare_encoder = RareLabelEncoder(tol=0.01, n_categories=1, variables=categorical_others)
target_encoder = OrdinalEncoder(encoding_method='ordered', variables=categorical_others)


# Feature engineering + scaling + lasso model 
model_pipeline = Pipeline([
    ('yr_transformer', yr_transformer),
    ('yr_dropper', yr_dropper),
    ('missing_indicator', missing_indicator),
    ('impute_mean', impute_mean),
    ('log_transformer', log_transformer),
    ('yeo_transformer', yeo_transformer),
    ('binarizer', binarizer),
    ('fill_missing', fill_missing),
    ('fill_frequent', fill_frequent),
    ('mapper_qual', mapper_qual),
    ('mapper_exposure', mapper_exposure),
    ('mapper_finish', mapper_finish),
    ('mapper_garage', mapper_garage),
    ('mapper_fence', mapper_fence),
    ('rare_encoder', rare_encoder),
    ('target_encoder', target_encoder),
    ('scaler', MinMaxScaler()),
    ('lasso', Lasso(alpha=0.001, random_state=RANDOM_STATE))
])

model_pipeline

## Training and prediction

This model can then be fitted directly with the training data being passed recursively along the pipeline steps. Here we have to start fresh so we perform again the split into train and validation sets as above.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    data.drop(['LogSalePrice', 'SalePrice'], axis=1), 
    data['LogSalePrice'], 
    test_size=0.1, 
    random_state=RANDOM_STATE,
)

X_train[categorical] = X_train[categorical].astype('O')
X_valid[categorical] = X_valid[categorical].astype('O')

model_pipeline.fit(X_train, y_train);

Note that inference can be performed in a single step:

In [ ]:
print('RMSE (train):', mean_squared_error(np.exp(model_pipeline.predict(X_train)), np.exp(y_train), squared=False))
print('R2   (train):', r2_score(np.exp(y_train), np.exp(model_pipeline.predict(X_train))))
print()
print('RMSE (valid):', mean_squared_error(np.exp(y_valid), np.exp(model_pipeline.predict(X_valid)), squared=False))
print('R2   (valid):', r2_score(np.exp(y_valid), np.exp(model_pipeline.predict(X_valid))))

Performing inference on new data:

In [ ]:
test_data = pd.read_csv('./data/house-prices-advanced-regression-techniques/test.csv')
X_test = test_data.drop(['Id'], axis=1)
X_test[categorical] = X_test[categorical].astype('O')

# Missing only on test time => skip
features = data.drop(['SalePrice', 'LogSalePrice'], axis=1).columns
train_no_missing = [v for v in features if v not in (categorical_na + numerical_na)]
X_test = X_test[X_test[train_no_missing].isna().sum(axis=1) == 0]

submission = np.exp(model_pipeline.predict(X_test))

Plotting distributions. Looks good!

In [ ]:
sns.histplot(np.exp(y_train), kde=True, stat='density', bins=50, label='train', color="C0")
sns.histplot(submission, kde=True, stat='density', bins=50, label='test', color="C1")
plt.xlabel("SalePrice")
plt.legend();

## Conclusion

Pipelines are **awesome**. We can use scikit-learn transformers, as well as custom transformers, as steps in our pipelines, making this technique very flexible. It is not done in this notebook, but even feature engineering decisions can be included in the pipeline by using flags or by introducing hyperparameters (e.g. we have the `tol` hyperparameter for the rare label threshold above). 

These flags and parameters can be part of cross-validation and optimization using hyperparameter optimization libraries such as [Optuna](https://particle1331.github.io/ok-transformer/nb/fundamentals/optuna.html) and experiment trackers such as [MLflow](https://particle1331.github.io/ok-transformer/nb/mlops/02-mlflow/notes.html). Finally, we have shown that the pipeline can be loaded to perform inference directly on new test data with minimal preprocessing.